# Heterophilous Graphs Preprocessing for QuVINE

This notebook converts the five Yandex Research heterophilous graph NPZ datasets into a QuVINE-compatible input layout without requiring changes to existing graph-loading scripts.

## Datasets

- `roman_empire`
- `amazon_ratings`
- `minesweeper`
- `tolokers`
- `questions`

Each `*.npz` contains:

- `edges`
- `node_features`
- `node_labels`
- `train_masks`
- `val_masks`
- `test_masks`

## Outputs

The notebook writes two separate output trees:

- `A_subsamples/` for repeated connected induced subgraphs of approximately 2K, 5K, and 10K nodes, with 30 repeats per size
- `B_full/` for full-graph exports that preserve the provided train/val/test splits

For compatibility with existing QuVINE job scripts, every graph bundle is saved as:

- `*.csv` edge list with columns `node1,node2`
- `*.json` metadata sidecar
- `*_node_labels.npy`
- `*_node_features.npy`
- `*_train_masks.npy`
- `*_val_masks.npy`
- `*_test_masks.npy`
- `*_node_index.csv` mapping exported node IDs to original IDs

## Notes

- Graph files remain CSV-based for downstream compatibility.
- Labels, features, and split masks are saved as sidecar NumPy files.
- For subsamples, masks are restricted to sampled nodes only.
- If a requested target size exceeds the graph size, the notebook skips that size for that dataset.

## 1. Imports

In [ ]:
from __future__ import annotations

import json
from collections import deque
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

## 2. Configuration

In [ ]:
DATASET_NAMES = [
    "roman_empire",
    "amazon_ratings",
    "minesweeper",
    "tolokers",
    "questions",
]

RAW_DIR = Path("../data/heterophilous_graphs/raw")
OUTPUT_ROOT = Path("../data/heterophilous_graphs/processed")
OUTPUT_A_DIR = OUTPUT_ROOT / "A_subsamples"
OUTPUT_B_DIR = OUTPUT_ROOT / "B_full"

DOWNLOAD_BASE = "https://raw.githubusercontent.com/yandex-research/heterophilous-graphs/main/data"

TARGET_SIZES = [2000, 5000, 10000]
N_SUBSAMPLES = 30
BASE_SEED = 42

RESTART_PROB = 0.15
FRONTIER_WIDTH = 256
DEGREE_WEIGHT_POWER = 0.5
LOCAL_BFS_EXPANSION = 8

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_A_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_B_DIR.mkdir(parents=True, exist_ok=True)

print("Raw dir:", RAW_DIR.resolve())
print("Output root:", OUTPUT_ROOT.resolve())

## 3. Download the NPZ files

In [ ]:
def ensure_npz_downloaded(dataset_name: str, raw_dir: Path = RAW_DIR, overwrite: bool = False) -> Path:
    raw_dir.mkdir(parents=True, exist_ok=True)
    target_path = raw_dir / f"{dataset_name}.npz"
    if target_path.exists() and not overwrite:
        print(f"Using existing file: {target_path}")
        return target_path

    url = f"{DOWNLOAD_BASE}/{dataset_name}.npz"
    print(f"Downloading {url} -> {target_path}")
    urlretrieve(url, target_path)
    return target_path


downloaded_paths = {name: ensure_npz_downloaded(name) for name in DATASET_NAMES}
downloaded_paths

## 4. Load helpers

In [ ]:
def _normalize_edges_array(edges: np.ndarray) -> np.ndarray:
    edges = np.asarray(edges)
    if edges.ndim != 2:
        raise ValueError(f"Expected 2D edge array, got shape {edges.shape}")
    if edges.shape[1] == 2:
        out = edges
    elif edges.shape[0] == 2:
        out = edges.T
    else:
        raise ValueError(f"Could not infer edge array orientation from shape {edges.shape}")
    return out.astype(np.int64, copy=False)


def ensure_2d_masks(mask_array: np.ndarray) -> np.ndarray:
    mask_array = np.asarray(mask_array)
    if mask_array.ndim == 1:
        mask_array = mask_array[:, None]
    elif mask_array.ndim == 2:
        pass
    else:
        raise ValueError(f"Expected 1D or 2D mask array, got shape {mask_array.shape}")
    return mask_array.astype(bool, copy=False)


def load_heterophilous_npz(npz_path: Path) -> Dict[str, np.ndarray]:
    with np.load(npz_path, allow_pickle=False) as data:
        payload = {key: data[key] for key in data.files}

    payload["edges"] = _normalize_edges_array(payload["edges"])
    payload["node_features"] = np.asarray(payload["node_features"])
    payload["node_labels"] = np.asarray(payload["node_labels"]).astype(np.int64, copy=False)
    payload["train_masks"] = ensure_2d_masks(payload["train_masks"])
    payload["val_masks"] = ensure_2d_masks(payload["val_masks"])
    payload["test_masks"] = ensure_2d_masks(payload["test_masks"])
    return payload


def build_graph_from_edges(edges: np.ndarray, n_nodes: int) -> nx.Graph:
    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    G.add_edges_from((int(u), int(v)) for u, v in edges if int(u) != int(v))
    return G


def summarize_dataset(payload: Dict[str, np.ndarray]) -> Dict[str, int]:
    n_nodes = int(payload["node_features"].shape[0])
    G = build_graph_from_edges(payload["edges"], n_nodes)
    return {
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "n_features": int(payload["node_features"].shape[1]),
        "n_classes": int(np.unique(payload["node_labels"]).size),
        "n_splits": int(payload["train_masks"].shape[1]),
        "n_components": int(nx.number_connected_components(G)),
    }

## 5. Quick dataset inspection

In [ ]:
payloads = {name: load_heterophilous_npz(path) for name, path in downloaded_paths.items()}
summary_df = pd.DataFrame(
    [{"dataset": name, **summarize_dataset(payload)} for name, payload in payloads.items()]
).sort_values("dataset")
summary_df

## 6. Sampling helpers for A (subsamples)

In [ ]:
def graph_degree_array(G: nx.Graph) -> np.ndarray:
    return np.array([d for _, d in G.degree()], dtype=float)


def materialize_undirected_simple_graph(G: nx.Graph) -> nx.Graph:
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from(G.edges(data=True))
    H.remove_edges_from(nx.selfloop_edges(H))
    return H


def induce_subgraph_by_nodes(G: nx.Graph, nodes: Iterable[int]) -> nx.Graph:
    node_set = set(nodes)
    H = nx.Graph()
    H.add_nodes_from(node_set)
    H.add_edges_from((u, v) for u, v in G.edges() if u in node_set and v in node_set)
    return H


def keep_largest_connected_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    lcc_nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(lcc_nodes).copy()


def weighted_choice_without_replacement(items, weights, k, rng):
    items = list(items)
    weights = np.asarray(weights, dtype=float)
    if len(items) == 0:
        return []
    if np.all(weights <= 0):
        weights = np.ones(len(items), dtype=float)
    weights = np.maximum(weights, 1e-12)
    probs = weights / weights.sum()
    k_eff = min(k, len(items))
    idx = rng.choice(len(items), size=k_eff, replace=False, p=probs)
    return [items[i] for i in idx]


def sample_anchor_node(G: nx.Graph, rng: np.random.Generator, power: float = 0.5):
    nodes = list(G.nodes())
    deg = np.array([G.degree(n) for n in nodes], dtype=float)
    weights = np.power(np.maximum(deg, 1.0), power)
    weights = weights / weights.sum()
    return nodes[int(rng.choice(len(nodes), p=weights))]


def sample_degree_targets(G: nx.Graph, sample_size: int, rng: np.random.Generator) -> np.ndarray:
    deg = graph_degree_array(G)
    if len(deg) == 0:
        return np.zeros(sample_size)
    return rng.choice(deg, size=sample_size, replace=True)


def connected_degree_aware_subgraph(
    G: nx.Graph,
    target_size: int,
    rng: np.random.Generator,
    restart_prob: float = 0.15,
    degree_weight_power: float = 0.5,
    frontier_width: int = 256,
    local_bfs_expansion: int = 8,
) -> nx.Graph:
    if target_size >= G.number_of_nodes():
        return materialize_undirected_simple_graph(G)

    anchor = sample_anchor_node(G, rng, power=degree_weight_power)
    selected = [anchor]
    selected_set = {anchor}
    frontier = set(G.adj[anchor].keys())
    degree_targets = sample_degree_targets(G, target_size, rng)
    target_ptr = 0

    while len(selected) < target_size:
        if not frontier or rng.random() < restart_prob:
            seed_from_selected = selected[int(rng.integers(0, len(selected)))]
            local_frontier = deque([seed_from_selected])
            steps = 0
            while local_frontier and steps < local_bfs_expansion:
                u = local_frontier.popleft()
                nbrs = list(G.adj[u].keys())
                rng.shuffle(nbrs)
                for v in nbrs:
                    if v not in selected_set:
                        frontier.add(v)
                        local_frontier.append(v)
                steps += 1

        if not frontier:
            remaining = list(set(G.nodes()) - selected_set)
            if not remaining:
                break
            candidate = remaining[int(rng.integers(0, len(remaining)))]
            frontier.add(candidate)

        frontier_list = list(frontier)
        if len(frontier_list) > frontier_width:
            frontier_list = weighted_choice_without_replacement(
                frontier_list,
                [max(G.degree(n), 1) for n in frontier_list],
                frontier_width,
                rng,
            )

        target_degree = degree_targets[min(target_ptr, len(degree_targets) - 1)]
        scores = []
        for node in frontier_list:
            deg = G.degree(node)
            internal_links = sum((nbr in selected_set) for nbr in G.adj[node].keys())
            degree_match = 1.0 / (1.0 + abs(deg - target_degree))
            score = 2.0 * internal_links + degree_match + 0.25 * np.log1p(deg)
            scores.append(score)

        scores = np.asarray(scores, dtype=float)
        if np.all(scores <= 0):
            scores = np.ones_like(scores)
        probs = scores / scores.sum()
        chosen = frontier_list[int(rng.choice(len(frontier_list), p=probs))]

        selected.append(chosen)
        selected_set.add(chosen)
        frontier.discard(chosen)
        frontier.update(v for v in G.adj[chosen].keys() if v not in selected_set)
        target_ptr += 1

    H = induce_subgraph_by_nodes(G, selected_set)
    H = materialize_undirected_simple_graph(H)
    H = keep_largest_connected_component(H)

    while H.number_of_nodes() < target_size and H.number_of_nodes() < G.number_of_nodes():
        current_nodes = set(H.nodes())
        boundary = set()
        for u in current_nodes:
            boundary.update(v for v in G.adj[u].keys() if v not in current_nodes)
        if not boundary:
            break
        boundary = list(boundary)
        rng.shuffle(boundary)
        needed = min(target_size - H.number_of_nodes(), len(boundary))
        current_nodes.update(boundary[:needed])
        H = keep_largest_connected_component(induce_subgraph_by_nodes(G, current_nodes))

    return H


def degree_histogram_distance(G_ref: nx.Graph, G_sub: nx.Graph, bins: int = 30) -> float:
    deg_ref = graph_degree_array(G_ref)
    deg_sub = graph_degree_array(G_sub)
    if len(deg_ref) == 0 or len(deg_sub) == 0:
        return float("inf")
    upper = max(float(deg_ref.max()), float(deg_sub.max()), 1.0)
    bin_edges = np.linspace(0.0, upper, bins + 1)
    h_ref, _ = np.histogram(deg_ref, bins=bin_edges, density=True)
    h_sub, _ = np.histogram(deg_sub, bins=bin_edges, density=True)
    return float(np.abs(h_ref - h_sub).sum())

## 7. Export helpers

In [ ]:
def save_graph_edgelist_csv(edges: np.ndarray, path: Path):
    df = pd.DataFrame(edges, columns=["node1", "node2"])
    df.to_csv(path, index=False)


def save_node_index_csv(exported_node_ids: np.ndarray, original_node_ids: np.ndarray, path: Path):
    df = pd.DataFrame({
        "export_node_id": exported_node_ids.astype(int),
        "original_node_id": original_node_ids.astype(int),
    })
    df.to_csv(path, index=False)


def make_export_bundle_arrays(
    original_edges: np.ndarray,
    node_features: np.ndarray,
    node_labels: np.ndarray,
    train_masks: np.ndarray,
    val_masks: np.ndarray,
    test_masks: np.ndarray,
    selected_nodes: np.ndarray,
) -> Dict[str, np.ndarray]:
    selected_nodes = np.asarray(selected_nodes, dtype=np.int64)
    selected_set = set(selected_nodes.tolist())
    old_to_new = {int(old): int(new) for new, old in enumerate(selected_nodes.tolist())}

    edge_mask = np.array(
        [(int(u) in selected_set) and (int(v) in selected_set) for u, v in original_edges],
        dtype=bool,
    )
    sub_edges = original_edges[edge_mask]
    remapped_edges = np.array([[old_to_new[int(u)], old_to_new[int(v)]] for u, v in sub_edges], dtype=np.int64)

    return {
        "selected_original_nodes": selected_nodes,
        "exported_node_ids": np.arange(len(selected_nodes), dtype=np.int64),
        "edges": remapped_edges,
        "node_features": node_features[selected_nodes],
        "node_labels": node_labels[selected_nodes],
        "train_masks": train_masks[selected_nodes],
        "val_masks": val_masks[selected_nodes],
        "test_masks": test_masks[selected_nodes],
    }


def write_bundle(
    bundle_dir: Path,
    stem: str,
    arrays: Dict[str, np.ndarray],
    metadata: Dict,
):
    bundle_dir.mkdir(parents=True, exist_ok=True)
    csv_path = bundle_dir / f"{stem}.csv"
    json_path = bundle_dir / f"{stem}.json"
    labels_path = bundle_dir / f"{stem}_node_labels.npy"
    features_path = bundle_dir / f"{stem}_node_features.npy"
    train_masks_path = bundle_dir / f"{stem}_train_masks.npy"
    val_masks_path = bundle_dir / f"{stem}_val_masks.npy"
    test_masks_path = bundle_dir / f"{stem}_test_masks.npy"
    node_index_path = bundle_dir / f"{stem}_node_index.csv"

    save_graph_edgelist_csv(arrays["edges"], csv_path)
    np.save(labels_path, arrays["node_labels"])
    np.save(features_path, arrays["node_features"])
    np.save(train_masks_path, arrays["train_masks"])
    np.save(val_masks_path, arrays["val_masks"])
    np.save(test_masks_path, arrays["test_masks"])
    save_node_index_csv(arrays["exported_node_ids"], arrays["selected_original_nodes"], node_index_path)

    metadata = dict(metadata)
    metadata.update({
        "csv_path": str(csv_path),
        "labels_path": str(labels_path),
        "features_path": str(features_path),
        "train_masks_path": str(train_masks_path),
        "val_masks_path": str(val_masks_path),
        "test_masks_path": str(test_masks_path),
        "node_index_path": str(node_index_path),
    })

    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "csv": csv_path,
        "json": json_path,
        "labels": labels_path,
        "features": features_path,
        "train_masks": train_masks_path,
        "val_masks": val_masks_path,
        "test_masks": test_masks_path,
        "node_index": node_index_path,
    }


def split_counts(mask_array: np.ndarray) -> List[int]:
    return [int(mask_array[:, i].sum()) for i in range(mask_array.shape[1])]

## 8. Export B: full graphs with original splits

In [ ]:
full_export_records = []

for dataset_name, payload in payloads.items():
    n_nodes = int(payload["node_features"].shape[0])
    arrays = make_export_bundle_arrays(
        original_edges=payload["edges"],
        node_features=payload["node_features"],
        node_labels=payload["node_labels"],
        train_masks=payload["train_masks"],
        val_masks=payload["val_masks"],
        test_masks=payload["test_masks"],
        selected_nodes=np.arange(n_nodes, dtype=np.int64),
    )

    stem = f"{dataset_name}_full"
    metadata = {
        "dataset_family": "heterophilous_graphs",
        "dataset_name": dataset_name,
        "network_id": stem,
        "type": "heterophilous_full",
        "export_mode": "B_full",
        "graph_name": stem,
        "n_nodes": int(arrays["node_features"].shape[0]),
        "n_edges": int(arrays["edges"].shape[0]),
        "n_features": int(arrays["node_features"].shape[1]),
        "n_classes": int(np.unique(arrays["node_labels"]).size),
        "n_splits": int(arrays["train_masks"].shape[1]),
        "train_counts_by_split": split_counts(arrays["train_masks"]),
        "val_counts_by_split": split_counts(arrays["val_masks"]),
        "test_counts_by_split": split_counts(arrays["test_masks"]),
        "original_num_nodes": n_nodes,
        "sampling": None,
    }

    paths = write_bundle(OUTPUT_B_DIR / dataset_name, stem, arrays, metadata)
    full_export_records.append({
        "dataset": dataset_name,
        "graph_id": stem,
        "n_nodes": metadata["n_nodes"],
        "n_edges": metadata["n_edges"],
        "csv_path": str(paths["csv"]),
        "json_path": str(paths["json"]),
    })

full_export_df = pd.DataFrame(full_export_records).sort_values("dataset")
full_export_df

## 9. Export A: repeated subsamples of 2K, 5K, 10K

In [ ]:
subsample_export_records = []

for dataset_name, payload in payloads.items():
    n_nodes = int(payload["node_features"].shape[0])
    G_full = build_graph_from_edges(payload["edges"], n_nodes)

    for target_size in TARGET_SIZES:
        if target_size > n_nodes:
            print(f"Skipping {dataset_name} target_size={target_size} because n_nodes={n_nodes}")
            continue

        for repeat_idx in range(N_SUBSAMPLES):
            seed = BASE_SEED + 1000 * repeat_idx + target_size
            rng = np.random.default_rng(seed)
            H = connected_degree_aware_subgraph(
                G_full,
                target_size=target_size,
                rng=rng,
                restart_prob=RESTART_PROB,
                degree_weight_power=DEGREE_WEIGHT_POWER,
                frontier_width=FRONTIER_WIDTH,
                local_bfs_expansion=LOCAL_BFS_EXPANSION,
            )

            selected_nodes = np.array(sorted(H.nodes()), dtype=np.int64)
            arrays = make_export_bundle_arrays(
                original_edges=payload["edges"],
                node_features=payload["node_features"],
                node_labels=payload["node_labels"],
                train_masks=payload["train_masks"],
                val_masks=payload["val_masks"],
                test_masks=payload["test_masks"],
                selected_nodes=selected_nodes,
            )

            stem = f"{dataset_name}_n{target_size}_rep{repeat_idx:02d}"
            degree_hist_l1 = degree_histogram_distance(G_full, H)
            metadata = {
                "dataset_family": "heterophilous_graphs",
                "dataset_name": dataset_name,
                "network_id": stem,
                "type": "heterophilous_subsample",
                "export_mode": "A_subsamples",
                "graph_name": stem,
                "n_nodes": int(arrays["node_features"].shape[0]),
                "n_edges": int(arrays["edges"].shape[0]),
                "n_features": int(arrays["node_features"].shape[1]),
                "n_classes": int(np.unique(arrays["node_labels"]).size),
                "n_splits": int(arrays["train_masks"].shape[1]),
                "train_counts_by_split": split_counts(arrays["train_masks"]),
                "val_counts_by_split": split_counts(arrays["val_masks"]),
                "test_counts_by_split": split_counts(arrays["test_masks"]),
                "original_num_nodes": n_nodes,
                "sampling": {
                    "target_size": int(target_size),
                    "effective_target_size": int(arrays["node_features"].shape[0]),
                    "repeat_idx": int(repeat_idx),
                    "seed": int(seed),
                    "degree_hist_l1": float(degree_hist_l1),
                    "restart_prob": RESTART_PROB,
                    "degree_weight_power": DEGREE_WEIGHT_POWER,
                    "frontier_width": FRONTIER_WIDTH,
                    "local_bfs_expansion": LOCAL_BFS_EXPANSION,
                },
            }

            paths = write_bundle(OUTPUT_A_DIR / dataset_name / f"n{target_size}", stem, arrays, metadata)
            subsample_export_records.append({
                "dataset": dataset_name,
                "target_size": int(target_size),
                "repeat_idx": int(repeat_idx),
                "effective_n_nodes": int(arrays["node_features"].shape[0]),
                "n_edges": int(arrays["edges"].shape[0]),
                "degree_hist_l1": float(degree_hist_l1),
                "csv_path": str(paths["csv"]),
                "json_path": str(paths["json"]),
            })

subsample_export_df = pd.DataFrame(subsample_export_records).sort_values(["dataset", "target_size", "repeat_idx"])
subsample_export_df.head()

## 10. Save manifest tables

In [ ]:
full_manifest_path = OUTPUT_B_DIR / "manifest_full.csv"
subsample_manifest_path = OUTPUT_A_DIR / "manifest_subsamples.csv"

full_export_df.to_csv(full_manifest_path, index=False)
subsample_export_df.to_csv(subsample_manifest_path, index=False)

print("Saved:", full_manifest_path)
print("Saved:", subsample_manifest_path)

## 11. Optional diagnostics

In [ ]:
subsample_export_df.groupby(["dataset", "target_size"])["degree_hist_l1"].agg(["mean", "std", "min", "max"]).reset_index()

## 12. Example: inspect one exported bundle

In [ ]:
example_row = full_export_df.iloc[0]
example_json = Path(example_row["json_path"])
with open(example_json) as f:
    example_metadata = json.load(f)
example_metadata